
Build encounter cards, compute E5 embeddings, and store them in Neo4j.


# Neo4j Векторный поиск
1. Connects to Neo4j using the configured URI/user/password.
2. Creates a vector index on `Encounter.embedding_e5` (индекс на уровне бд по свойству ...)
3. Builds text cards for each `Encounter` (cities/airports/countries/documents/topics).
4. Computes E5 embeddings for those cards.
5. Writes embeddings back to Neo4j as a node property.


# Создание эмбедингов

In [24]:
# 1 создание эмбедингов для Encounters через карточки + инфа о связи bva-paris и аналоги
# Подключение к БД
from neo4j import GraphDatabase

NEO4J_URI = "neo4j://localhost:7689"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"
NEO4J_DB = "neo4j"

EMBEDDING_PROP = "embedding_e5"
VECTOR_INDEX_NAME = "encounter_embedding_e5"
VECTOR_DIM = 768  # e5-base-v2

def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Создание векторного индекса по свойству Encounter.embedding_e5 
def ensure_vector_index(driver):
    with driver.session(database=NEO4J_DB) as session:
        session.run(
            f"""
            CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
            FOR (e:Encounter) ON (e.{EMBEDDING_PROP})
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {VECTOR_DIM}, `vector.similarity_function`: 'cosine' }} }}
            """
        ).consume()
from typing import Dict, List

# Map smaller/metro airports to their larger city for embedding context
AIRPORT_ALIAS_CITY = {
    "airport_bva": "Paris",
    "airport_cdg": "Paris",
    "airport_ory": "Paris",
    "airport_sdg": "Paris",
    "airport_memmingen": "Munich",
    "airport_malpensa": "Milan",
    "airport_marco_polo": "Venice",
    "airport_fiumicino": "Rome",
    "airport_tegel": "Berlin",
    "airport_schonefeld": "Berlin",
    "airport_pisa": "Florence",
}

def fetch_encounter_cards(driver) -> Dict[str, str]:
    query = '''
    MATCH (e:Encounter)
    OPTIONAL MATCH (e)-[:atCity]->(city:City)
    OPTIONAL MATCH (e)-[:atAirport]->(airport:Airport)
    OPTIONAL MATCH (e)-[:atCountry]->(country:Country)
    OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
    RETURN e.key AS ekey,
           collect(DISTINCT city.key) AS cities,
           collect(DISTINCT airport.key) AS airports,
           collect(DISTINCT country.key) AS countries,
           collect(DISTINCT di.documentType) AS doc_types,
           collect(DISTINCT q.topicKey) AS topics
    '''
    cards = {}
    with driver.session(database=NEO4J_DB) as session:
        for r in session.run(query):
            ekey = r["ekey"]
            if not ekey:
                continue
            parts = [f"Encounter: {ekey}"]
            if r["cities"]:
                parts.append("Cities: " + ", ".join(sorted([c for c in r["cities"] if c])))
            if r["airports"]:
                airports = sorted([a for a in r["airports"] if a])
                parts.append("Airports: " + ", ".join(airports))
                alias_cities = sorted({AIRPORT_ALIAS_CITY[a] for a in airports if a in AIRPORT_ALIAS_CITY})
                if alias_cities:
                    parts.append("AliasCities: " + ", ".join(alias_cities))
            if r["countries"]:
                parts.append("Countries: " + ", ".join(sorted([c for c in r["countries"] if c])))
            if r["doc_types"]:
                parts.append("Documents: " + ", ".join(sorted([d for d in r["doc_types"] if d])))
            if r["topics"]:
                parts.append("Topics: " + ", ".join(sorted([t for t in r["topics"] if t])))
            cards[ekey] = " | ".join(parts)
    return cards

from typing import Iterable
try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError("Install sentence-transformers to compute E5 embeddings") from e

def embed_texts_e5(model: SentenceTransformer, texts: Iterable[str], is_query: bool = False) -> List[List[float]]:
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + t for t in texts]
    vecs = model.encode(texts, normalize_embeddings=True)
    return [v.tolist() for v in vecs]

def build_embeddings_for_encounters(cards: Dict[str, str]):
    model = SentenceTransformer("intfloat/e5-base-v2")
    keys = list(cards.keys())
    texts = [cards[k] for k in keys]
    vectors = embed_texts_e5(model, texts, is_query=False)
    return keys, vectors
def write_embeddings(driver, keys: List[str], vectors: List[List[float]]):
    with driver.session(database=NEO4J_DB) as session:
        for k, v in zip(keys, vectors):
            session.run(
                "MATCH (e:Encounter {key:$k}) SET e.%s = $v" % EMBEDDING_PROP,
                k=k, v=v
            ).consume()
# ___________________________________________________________________________________________________________________
# ПАЙПЛАЙН создания и записи всех ембедингов для Encounters
with get_driver() as driver:
    ensure_vector_index(driver)
    cards = fetch_encounter_cards(driver)
    keys, vecs = build_embeddings_for_encounters(cards)
    write_embeddings(driver, keys, vecs)
    print(f"Wrote embeddings for {len(keys)} encounters")


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 12203.08it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wrote embeddings for 154 encounters
